<a href="https://colab.research.google.com/github/FatimaMirandap/ML-Datostadamx2026/blob/main/mappingML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mapa de la investigación con ciencia de datos y machine learning

Versión 1 del cuaderno. Material del evento DatostadaMx 2026, "Ciencia de Datos y Machine Learning en la investigación: lo que ya es posible y lo que viene". Creado por Fátima Miranda Pestaña.

**Aviso.** Existe una mejora en desarrollo: una versión 2 del cuaderno y del mapa en HTML, con filtro por año y animación de la evolución en el tiempo. Estará disponible en unos días en este mismo repositorio. Esta versión 1 seguirá funcionando.

## Qué hace este cuaderno

Cuenta, por institución, cuántas publicaciones de **Nature, Science, Scientific Data y arXiv** mencionan ciencia de datos o machine learning en su título o resumen, dentro del período elegido. Para cada institución agrega tipo, país, ciudad y coordenadas. El resultado son dos archivos, `instituciones.csv` y `meta.json`, que alimentan el mapa interactivo de `docs/index.html`.

## Contenido

1. Configuración
2. Funciones de apoyo
3. Fuentes
4. Criterio de machine learning
5. Revisión de una muestra
6. Líneas de investigación
7. Descarga agregada por institución
8. Tipo de institución, país y coordenadas
9. Vista previa
10. Guardar los archivos del mapa
11. Exportar la vista previa (opcional)
12. Publicar el mapa
13. Solución de problemas
14. Trabajo futuro

## Requisitos

- Una cuenta de Google, para usar Google Colab.
- Una cuenta gratuita de OpenAlex, para obtener una clave de API.
- Aproximadamente media hora la primera vez.

## Inicio rápido

1. Obtén tu clave de OpenAlex y guárdala como secreto de Colab (sección siguiente).
2. Ejecuta las celdas en orden. Con `DRY_RUN = True`, la celda de descarga solo estima cuántas llamadas haría y cuánto costaría.
3. Cuando la estimación te parezca razonable, cambia `DRY_RUN` a `False` en la celda de configuración y vuelve a ejecutar desde la celda de descarga.
4. Descarga `instituciones.csv` y `meta.json` y colócalos en `docs/data/` del repositorio.

## Qué es una "rebanada"

La descarga se divide en rebanadas: una por cada combinación de fuente y línea de investigación (4 fuentes por 26 campos son 104 rebanadas). Para cada una se pide a OpenAlex el conteo de obras agrupado por institución, sin descargar las obras.

## Límites que conviene mencionar al presentar el mapa

- La unidad es la publicación, no el proyecto de investigación: muchos proyectos no publican en estas revistas.
- "Tiene machine learning" se decide por palabras clave en título y resumen, así que hay falsos positivos y falsos negativos. La sección 5 permite revisar una muestra.
- Cada obra se cuenta una vez por institución. Una obra con autores de tres instituciones suma en las tres, de modo que el total son participaciones institucionales, no obras.
- Science no ofrece una API pública. Aquí se consulta únicamente su metadato abierto a través de OpenAlex.
- El tipo, el país y la ubicación de cada institución provienen de OpenAlex y pueden tener errores.

## Antes de empezar: clave de OpenAlex y secretos de Colab

OpenAlex exige una clave de API. Es gratuita y da un presupuesto de 1 USD al día, que se reinicia a medianoche UTC (las 18:00 en la Ciudad de México y Mérida). Sin clave, el presupuesto es de 0.01 USD al día, que no alcanza para este cuaderno.

1. Crea una cuenta gratuita en openalex.org.
2. Copia tu clave desde la página de configuración de la API de tu cuenta (openalex.org/settings/api).
3. En Colab, abre el panel izquierdo, pulsa el ícono de la llave (Secretos) y agrega un secreto nuevo con el nombre `OPENALEX_API_KEY` y tu clave como valor.
4. Activa el interruptor de acceso del cuaderno a ese secreto.

Nunca escribas la clave dentro de una celda: quedaría visible si compartes o publicas el cuaderno.

**Costos de referencia.** Las llamadas con búsqueda de texto cuestan unos 0.001 USD y las llamadas de lista unos 0.0001 USD. Con 104 rebanadas y unas dos llamadas por rebanada, la descarga completa cuesta alrededor de 0.2 a 0.5 USD, dentro del presupuesto diario gratuito. Los precios pueden cambiar: la página de precios de OpenAlex es la referencia.

## 1. Configuración

Esta celda importa las librerías y define los parámetros que puedes cambiar.

| Parámetro | Qué controla |
| --- | --- |
| `EMAIL` | Correo de contacto opcional que se envía a OpenAlex. Se recomienda poner uno propio. No lo publiques si el repositorio es público. |
| `PERIODO` | Años de publicación, en formato `AAAA-AAAA`. |
| `DRY_RUN` | `True` solo estima llamadas y costo; `False` descarga los datos. |
| `MAX_PAGINAS` | Máximo de páginas de 200 instituciones por rebanada. |
| `PRESUPUESTO_USD` | Tope diario con el que se compara la estimación. El tope real del plan gratuito de OpenAlex es 1 USD al día. |

La clave de OpenAlex se lee del secreto de Colab `OPENALEX_API_KEY`. Si ejecutas el cuaderno fuera de Colab, se lee de la variable de entorno con el mismo nombre.

In [ ]:
import os, time, json, math, random, re
import requests
import pandas as pd

EMAIL = ""
PERIODO = "2020-2026"
DRY_RUN = True
MAX_PAGINAS = 5
PRESUPUESTO_USD = 1.0

try:
    from google.colab import userdata
    OPENALEX_API_KEY = userdata.get("OPENALEX_API_KEY")
except Exception:
    OPENALEX_API_KEY = os.environ.get("OPENALEX_API_KEY")

if OPENALEX_API_KEY:
    print("Clave de OpenAlex: encontrada")
else:
    print("Clave de OpenAlex: no encontrada. El presupuesto será muy bajo y las llamadas fallarán.")

## 2. Funciones de apoyo

- `oa(ruta, es_busqueda, **params)` hace una llamada a la API de OpenAlex. Agrega la clave y el correo, espera 0.15 segundos entre llamadas para no exceder el límite de 100 solicitudes por segundo y reintenta hasta 5 veces cuando la API responde 429 (demasiadas solicitudes) o un error temporal del servidor (500, 502 o 503), esperando cada vez el doble. Cuenta las llamadas en el diccionario `llamadas`.
- `costo_estimado(n_llamadas, n_busquedas)` calcula el costo aproximado en dólares: unos 0.0001 USD por llamada de lista y unos 0.001 USD por llamada con búsqueda de texto (precios de OpenAlex de febrero de 2026).

**Nota sobre el error 429.** OpenAlex usa el mismo código 429 para un freno breve y para un presupuesto diario agotado. Si el cuaderno falla con el mensaje "Falló works" después de muchas llamadas, lo más probable es que el presupuesto del día se haya agotado. Espera al reinicio de medianoche UTC y vuelve a ejecutar.

In [ ]:
BASE = "https://api.openalex.org"
llamadas = {"n": 0, "busqueda": 0}

def oa(ruta, es_busqueda=False, **params):
    params = {k: v for k, v in params.items() if v is not None}
    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY
    if EMAIL:
        params["mailto"] = EMAIL

    for intento in range(5):
        r = requests.get(f"{BASE}/{ruta}", params=params, timeout=60)
        if r.status_code == 200:
            llamadas["n"] += 1
            llamadas["busqueda"] += int(es_busqueda)
            time.sleep(0.15)
            return r.json()
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** intento)
            continue
        print("Error", r.status_code, r.text[:300])
        r.raise_for_status()
    raise RuntimeError(f"Falló {ruta}")

def costo_estimado(n_llamadas, n_busquedas):
    return (n_llamadas - n_busquedas) * 0.0001 + n_busquedas * 0.001

## 3. Fuentes: encuentra el identificador de cada revista

OpenAlex identifica cada revista o repositorio con un código que empieza con la letra S. Esta celda busca cada fuente por su nombre, muestra las cinco mejores coincidencias y elige la que tiene exactamente el mismo nombre.

Revisa la salida: cada fuente elegida debe ser la correcta. Si alguna no lo es, escribe su identificador a mano en el diccionario `FUENTES_ID` (por ejemplo `FUENTES_ID = {"Science": "S3880285"}`) antes de ejecutar la celda, o edita el diccionario `FUENTES` para cambiar el nombre de búsqueda. Para agregar otra revista, añade una entrada a `FUENTES`.

In [ ]:
FUENTES = {"Nature": "Nature", "Science": "Science",
           "Scientific Data": "Scientific Data", "arXiv": "arXiv"}
FUENTES_ID = {}

for etiqueta, nombre in FUENTES.items():
    j = oa("sources", search=nombre, per_page=5, es_busqueda=True,
           select="id,display_name,issn_l,works_count,type,host_organization_name")
    print(f"\n== {etiqueta} ==")
    for s in j.get("results", []):
        print(f"  {s.get('id').split('/')[-1]} | {s.get('display_name')} | {s.get('host_organization_name')} | {s.get('works_count')}")

    exactas = [s for s in j.get("results", []) if s.get("display_name", "").lower() == nombre.lower()]
    elegida = exactas[0] if exactas else j.get("results", [{}])[0]

    if elegida.get("id"):
        FUENTES_ID.setdefault(etiqueta, elegida["id"].split("/")[-1])

print("\nElegidas:", FUENTES_ID)

## 4. Criterio de machine learning

Una obra "tiene ciencia de datos o machine learning" si su título o su resumen contienen alguno de los términos de `TERMINOS_ML`. Se evita el término "neural network" a secas porque en neurociencia se refiere a redes biológicas.

- `ML_QUERY` une los términos con OR en una sola búsqueda.
- `TIPOS` limita las obras a artículos y preprints.
- `filtro_base(sid, extra)` arma el filtro de OpenAlex para una fuente. Usa `locations.source.id` y no `primary_location.source.id`, para no excluir preprints de arXiv que ya fueron publicados en otra revista.
- `total_sin_ml(sid)` es el mismo filtro sin el criterio de machine learning, para calcular qué porcentaje de las obras de cada fuente lo cumple.

La tabla resultante, `resumen_fuentes`, muestra el total de obras, las que cumplen el criterio y su porcentaje. Puedes cambiar los términos y volver a ejecutar esta celda hasta que el criterio te convenza.

In [ ]:
TERMINOS_ML = [
    '"machine learning"', '"deep learning"', '"data science"',
    '"artificial neural network"', '"convolutional neural network"',
    '"large language model"', '"foundation model"', '"random forest"',
    '"gradient boosting"', '"supervised learning"'
]

ML_QUERY = "(" + " OR ".join(TERMINOS_ML) + ")"
TIPOS = "article|preprint"

def filtro_base(sid, extra=""):
    f = f"locations.source.id:{sid},publication_year:{PERIODO},type:{TIPOS},title_and_abstract.search:{ML_QUERY}"
    return f + ("," + extra if extra else "")

def total_sin_ml(sid):
    return f"locations.source.id:{sid},publication_year:{PERIODO},type:{TIPOS}"

filas = []
for etiqueta, sid in FUENTES_ID.items():
    todas = oa("works", filter=total_sin_ml(sid), per_page=1, select="id")["meta"]["count"]
    con_ml = oa("works", filter=filtro_base(sid), per_page=1, select="id", es_busqueda=True)["meta"]["count"]
    filas.append({"fuente": etiqueta, "obras": todas, "con_ML": con_ml, "porcentaje": round(100 * con_ml / max(todas, 1), 1)})

resumen_fuentes = pd.DataFrame(filas)
display(resumen_fuentes)

## 5. Revisa una muestra a mano

Para validar el criterio, esta celda imprime 12 títulos al azar por fuente (con una semilla fija para que sea reproducible). Léelos: si aparecen temas que no usan machine learning en el sentido que buscas, ajusta `TERMINOS_ML` en la sección 4 y vuelve a ejecutar las secciones 4 y 5.

In [ ]:
for etiqueta, sid in FUENTES_ID.items():
    j = oa("works", filter=filtro_base(sid), sample=12, seed=7, per_page=12, select="id,title,publication_year", es_busqueda=True)
    print(f"\n== {etiqueta} ==")
    for w in j.get("results", []):
        print("  ", w.get("publication_year", ""), "|", (w.get("title") or "")[:110])

## 6. Líneas de investigación

Como "línea de investigación" se usan los **campos** de OpenAlex, que son 26 áreas amplias (por ejemplo Ciencias de la computación, Medicina o Ciencias de la Tierra y planetarias). La celda los descarga y guarda un diccionario con el identificador y el nombre de cada campo en `CAMPOS`.

In [ ]:
campos = oa("fields", per_page=50, select="id,display_name")["results"]
CAMPOS = {c["id"].split("/")[-1]: c["display_name"] for c in campos}
print(len(CAMPOS), "campos")
CAMPOS

## 7. Descarga agregada por institución

La celda calcula primero cuántas rebanadas hay (fuentes por campos), cuántas llamadas harían falta y cuánto costaría. La estimación supone tres páginas por rebanada, que es un valor conservador. Si supera `PRESUPUESTO_USD`, avisa.

- `grupos_por_institucion(filtro)` pide a OpenAlex el conteo de obras agrupado por institución, página por página, hasta `MAX_PAGINAS`. Se detiene si una página llega vacía, para evitar ciclos infinitos.
- Con `DRY_RUN = True` solo se imprime la estimación y `df` queda vacío, de modo que las celdas siguientes no fallan.
- Con `DRY_RUN = False` se descargan los datos y se guardan en `df`, con una fila por institución, fuente y campo.

Esta versión no guarda su avance: si se interrumpe, hay que volver a ejecutar la celda desde el principio. La descarga completa es corta (104 rebanadas), así que normalmente no hace falta.

In [ ]:
rebanadas = [(e, sid, cid, cn) for e, sid in FUENTES_ID.items() for cid, cn in CAMPOS.items()]
est_llamadas = len(rebanadas) * 3
est_costo = costo_estimado(est_llamadas, est_llamadas)
print(f"{len(rebanadas)} rebanadas, ~{est_llamadas} llamadas, costo estimado ~{est_costo:.2f} USD")

if est_costo > PRESUPUESTO_USD:
    print("Aviso: supera tu presupuesto diario; reduce MAX_PAGINAS o los campos.")

def grupos_por_institucion(filtro):
    salida, cursor = [], "*"
    for _ in range(MAX_PAGINAS):
        j = oa("works", filter=filtro, group_by="authorships.institutions.id", per_page=200, cursor=cursor, es_busqueda=True)

        grupos = j.get("group_by", [])
        if not grupos:
            break

        salida += grupos
        cursor = j.get("meta", {}).get("next_cursor")
        if not cursor:
            break
    return salida

resultados = []
if not DRY_RUN:
    for i, (etiqueta, sid, cid, cn) in enumerate(rebanadas, 1):
        g = grupos_por_institucion(filtro_base(sid, f"primary_topic.field.id:{cid}"))
        for x in g:
            if x.get("key") and x.get("count"):
                resultados.append({"institution_id": x["key"].split("/")[-1], "institution": x.get("key_display_name"),
                                   "source_group": etiqueta, "field": cn, "works": x["count"]})
        if i % 10 == 0:
            print(i, "de", len(rebanadas), "| llamadas:", llamadas["n"], "| filas:", len(resultados))

    df = pd.DataFrame(resultados)
    print(df.shape)
else:
    print("DRY_RUN activo: cambia DRY_RUN a False en la celda de configuración para descargar.")
    df = pd.DataFrame()

## 8. Tipo de institución, país y coordenadas

Para cada institución encontrada se consulta a OpenAlex su tipo (educación, empresa, gobierno, organización sin fines de lucro, salud, instalación científica, archivo u otro), su país, su ciudad y sus coordenadas. Las consultas se hacen en lotes de 50 identificadores.

Las filas de instituciones sin coordenadas se excluyen, porque no se pueden ubicar en el mapa; la celda imprime cuántas son. Después muestra cuántas filas hay por tipo de institución.

In [ ]:
if not DRY_RUN and not df.empty:
    ids = sorted(df["institution_id"].unique())
    info = {}
    for k in range(0, len(ids), 50):
        lote = ids[k:k + 50]
        j = oa("institutions", filter="openalex:" + "|".join(lote), per_page=50, select="id,display_name,type,country_code,geo")
        for it in j.get("results", []):
            g = it.get("geo") or {}
            info[it["id"].split("/")[-1]] = {"type": it.get("type"), "country_code": it.get("country_code"),
                                             "city": g.get("city"), "lat": g.get("latitude"), "lon": g.get("longitude")}

    meta_inst = pd.DataFrame.from_dict(info, orient="index").reset_index().rename(columns={"index": "institution_id"})
    df = df.merge(meta_inst, on="institution_id", how="left")
    sin_coord = df["lat"].isna().sum()
    print(f"Filas sin coordenadas (se excluyen del mapa): {sin_coord} de {len(df)}")
    df = df.dropna(subset=["lat", "lon"])
    print(df["type"].value_counts(dropna=False))

## 9. Vista previa en Colab

Dibuja un mapa interactivo con Plotly para comprobar que los datos tienen sentido antes de publicarlos. Cada círculo es una institución, su tamaño es el número de obras y su color es el tipo de institución. Requiere una versión reciente de Plotly (5.24 o posterior), que Colab ya incluye.

In [ ]:
if not DRY_RUN and not df.empty:
    import plotly.express as px
    agg = (df.groupby(["institution_id", "institution", "type", "country_code", "city", "lat", "lon"], as_index=False)["works"].sum())
    fig = px.scatter_map(agg, lat="lat", lon="lon", size="works", color="type", hover_name="institution",
                         hover_data=["country_code", "city", "works"], map_style="carto-positron", zoom=1, size_max=30)
    fig.show()

## 10. Guardar los archivos del mapa

Genera dos archivos:

- `instituciones.csv`, con una fila por institución, fuente y campo, y las columnas `institution_id`, `institution`, `type`, `country_code`, `city`, `lat`, `lon`, `source_group`, `field` y `works`.
- `meta.json`, con la fecha de extracción, el período, las fuentes, el criterio de machine learning, los tipos de obra, el catálogo y el resumen por fuente. El mapa usa este archivo para el pie de página.

Los archivos se descargan a tu computadora. Si el navegador bloquea la descarga, quedan en el panel de archivos de Colab (ícono de carpeta, en el panel izquierdo), desde donde puedes descargarlos a mano. Fuera de Colab se guardan en la carpeta donde ejecutas el cuaderno.

Después colócalos en la carpeta `docs/data/` del repositorio.

In [ ]:
if not DRY_RUN and not df.empty:
    from datetime import date
    columnas = ["institution_id", "institution", "type", "country_code", "city", "lat", "lon", "source_group", "field", "works"]
    df[columnas].to_csv("instituciones.csv", index=False)

    meta = {"fecha_de_corte": date.today().isoformat(), "periodo": PERIODO, "fuentes": FUENTES_ID,
            "criterio_ml": ML_QUERY, "tipos_de_obra": TIPOS, "catalogo": "OpenAlex",
            "resumen_fuentes": resumen_fuentes.to_dict(orient="records")}

    with open("meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    try:
        from google.colab import files
        files.download("instituciones.csv")
        files.download("meta.json")
    except Exception:
        print("Descarga automática no disponible. Los archivos están en la carpeta de trabajo:", os.getcwd())

## 11. Exportar la vista previa (opcional)

Guarda la vista previa de la sección 9 como una página HTML independiente, `mapa_interactivo.html`. Es distinta del mapa final de `docs/index.html`: no tiene filtros ni pie de página. Solo funciona si ya ejecutaste la sección 9.

In [ ]:
if "fig" in globals():
    fig.write_html("mapa_interactivo.html")
    print("Guardado: mapa_interactivo.html")
else:
    print("Aún no hay vista previa: ejecuta primero la sección 9 con DRY_RUN = False.")

## 12. Publicar el mapa

1. Copia `instituciones.csv` y `meta.json` a la carpeta `docs/data/` del repositorio.
2. Sube los cambios a GitHub. Hay instrucciones desde la consola en `TUTORIAL_GITHUB.md`, y el README explica cómo activar GitHub Pages y cómo incrustar el mapa en otra página.
3. Para probarlo en tu computadora, ejecuta `python -m http.server 8000` dentro de la carpeta `docs` y abre `http://localhost:8000`. Abrir `index.html` con doble clic no funciona, porque el navegador bloquea la lectura de los archivos de datos.

## 13. Solución de problemas

| Síntoma | Causa probable | Qué hacer |
| --- | --- | --- |
| "Clave de OpenAlex: no encontrada" | El secreto no existe, tiene otro nombre o el cuaderno no tiene acceso | Revisa el nombre `OPENALEX_API_KEY` y que el interruptor de acceso esté activado |
| Error 401 o 403 | Clave incorrecta | Copia de nuevo la clave desde tu cuenta de OpenAlex |
| "Falló works" después de muchas llamadas | Presupuesto diario agotado (error 429) | Espera al reinicio de medianoche UTC y vuelve a ejecutar |
| Error 400 con "Invalid filter" | Un filtro está mal escrito | Revisa `filtro_base` y los términos de `TERMINOS_ML` |
| `df` vacío y solo aparece la estimación | `DRY_RUN` está en `True` | Cámbialo a `False` y vuelve a ejecutar desde la sección 7 |
| El mapa muestra pocas instituciones | `MAX_PAGINAS` es bajo o hay filas sin coordenadas | Sube `MAX_PAGINAS` y revisa el aviso de la sección 8 |
| La vista previa no abre | Versión antigua de Plotly | Actualiza Plotly o usa `px.scatter_mapbox` |

## 14. Trabajo futuro

Versión 2, en desarrollo, que estará disponible en unos días:

- Descarga por año, que permitirá un filtro por año y ver la evolución en el tiempo con una animación.
- Descarga con reanudación, para continuar donde se quedó cuando se agota el presupuesto diario de OpenAlex.
- Mejoras del mapa en HTML.

Ideas posteriores:

- Añadir la API de Springer Nature (Nature y Scientific Data) para revisar resúmenes y materias de una muestra.
- Añadir Crossref como segundo catálogo para Science (metadatos y financiadores).